# BirdCLEF+ 2026 — Training Pipeline (Phase 4 + Phase 6)

This notebook produces the **LB-best 0.842 model**: a 10-model ensemble combining
two diversity strategies on top of Perch v2 spatial embeddings.

```
audio (5s) → Perch (spatial_embedding) → mean-pool freq → (16, 1536)
                                              ↓ SED head (LayerNorm + bottleneck + att/cla)
                                          clip logits (234)
```

We train **10 SED heads total**:
- **Phase 4**: 5 different random seeds on a fixed train/val split → `model_v8_seed{42..46}.pt`
- **Phase 6**: 5 K-folds on different train/val file splits → `model_v9_fold{0..4}.pt`

At inference, we average logits across all 10 → combined ensemble.

**Prereq**: `03_perch_embed.ipynb` must have run (needs `spatial_emb_clips.npy` + `spatial_emb_ss.npy`).
**Total runtime**: ~20-30 min on Apple Silicon MPS.

**Trajectory of LB scores**: 0.497 → 0.714 (Phase 0) → 0.772 (Phase 1) → 0.836 (Phase 3) → 0.839 (Phase 4 alone or Phase 6 alone) → **0.842 (combined)**.


## 1. Setup


In [ ]:
import os, ast, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, ConcatDataset
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("Device:", DEVICE, "  Torch:", torch.__version__)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EMBED_DIR    = PROJECT_ROOT / "embeddings"
CKPT_DIR     = PROJECT_ROOT / "checkpoints"
CKPT_DIR.mkdir(exist_ok=True)

for f in ["spatial_emb_clips.npy", "clip_index.csv", "spatial_emb_ss.npy", "ss_window_index.csv"]:
    assert (EMBED_DIR / f).exists(), f"Missing {EMBED_DIR / f} — run 03_perch_embed.ipynb first"

SEED         = 42
EMBED_DIM    = 1536
N_TIMESTEPS  = 16
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)


## 2. Load spatial embeddings + metadata


In [ ]:
clip_spatial = np.load(EMBED_DIR / "spatial_emb_clips.npy")    # (35549, 16, 1536)
ss_spatial   = np.load(EMBED_DIR / "spatial_emb_ss.npy")        # (1478, 16, 1536)
clip_index   = pd.read_csv(EMBED_DIR / "clip_index.csv")
clip_index["primary_label"] = clip_index["primary_label"].astype(str)
ss_index     = pd.read_csv(EMBED_DIR / "ss_window_index.csv")

print(f"Clip spatial:  {clip_spatial.shape}  {clip_spatial.nbytes/1e9:.1f} GB")
print(f"SS spatial:    {ss_spatial.shape}")

species      = sorted(clip_index["primary_label"].unique())
label_to_idx = {s: i for i, s in enumerate(species)}
NUM_CLASSES  = len(species)
print(f"NUM_CLASSES = {NUM_CLASSES}")


## 3. Label parsers and dataset class

- `clip_label_vec` — multi-hot from `primary_label` + `secondary_labels` (Python-list string).
- `ss_label_vec` — multi-hot from semicolon-separated species in soundscape labels.
- `SpatialEmbDataset` — returns `(spatial_embedding (16, 1536), multi_hot_label (234))`.


In [ ]:
def parse_secondary_labels(s):
    if pd.isna(s) or s in ("", "[]"):
        return []
    try:
        return list(ast.literal_eval(s))
    except (ValueError, SyntaxError):
        return []


def clip_label_vec(row, num_classes):
    vec = torch.zeros(num_classes, dtype=torch.float32)
    pl = str(row["primary_label"])
    if pl in label_to_idx:
        vec[label_to_idx[pl]] = 1.0
    for sp in parse_secondary_labels(row.get("secondary_labels", "[]")):
        sp = str(sp)
        if sp in label_to_idx:
            vec[label_to_idx[sp]] = 1.0
    return vec


def ss_label_vec(row, num_classes):
    vec = torch.zeros(num_classes, dtype=torch.float32)
    for sp in str(row["primary_label"]).split(";"):
        sp = sp.strip()
        if sp in label_to_idx:
            vec[label_to_idx[sp]] = 1.0
    return vec


class SpatialEmbDataset(Dataset):
    def __init__(self, spatial, index_df, label_fn):
        self.spatial  = spatial
        self.index_df = index_df.reset_index(drop=True)
        self.label_fn = label_fn

    def __len__(self):
        return len(self.spatial)

    def __getitem__(self, i):
        emb   = torch.from_numpy(self.spatial[i].copy())   # (16, 1536)
        label = self.label_fn(self.index_df.iloc[i], NUM_CLASSES)
        return emb, label


## 4. SED head architecture

```
spatial (B, 16, 1536)
  → LayerNorm
  → Linear(1536 → 512) + ReLU + Dropout
  (B, 16, 512), transposed to (B, 512, 16)
  → two parallel Conv1d(512 → 234, kernel=1)
       att_raw                 cla
  → softmax(tanh(att_raw)) → attention weights (B, 234, 16)
  → clip_logits = sum(attention × cla, dim=time)  → (B, 234)
```

The attention mechanism learns "which of the 16 timesteps matters for each species."


In [ ]:
class PerchSEDHead(nn.Module):
    def __init__(self, embed_dim=1536, hidden_dim=512, num_classes=234, dropout=0.3):
        super().__init__()
        self.norm = nn.LayerNorm(embed_dim)
        self.bottleneck = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.att = nn.Conv1d(hidden_dim, num_classes, kernel_size=1)
        self.cla = nn.Conv1d(hidden_dim, num_classes, kernel_size=1)
        nn.init.xavier_uniform_(self.att.weight)
        nn.init.xavier_uniform_(self.cla.weight)
        nn.init.zeros_(self.att.bias)
        nn.init.zeros_(self.cla.bias)

    def forward(self, x):
        # x: (B, T=16, D=1536)
        x = self.norm(x)
        x = self.bottleneck(x)        # (B, T, hidden)
        x = x.transpose(1, 2)         # (B, hidden, T)
        att_raw = self.att(x)         # (B, num_classes, T)
        cla     = self.cla(x)         # (B, num_classes, T)
        att     = torch.softmax(torch.tanh(att_raw), dim=2)
        return (att * cla).sum(dim=2)   # (B, num_classes)


_test = PerchSEDHead(EMBED_DIM, num_classes=NUM_CLASSES).to(DEVICE)
_x = torch.randn(2, N_TIMESTEPS, EMBED_DIM, device=DEVICE)
print(f"sample forward: input {tuple(_x.shape)} → output {tuple(_test(_x).shape)}")
print(f"Params: {sum(p.numel() for p in _test.parameters())/1e6:.2f}M")


## 5. Metric + mixup + train/validate helpers

- `macro_auc` — matches Kaggle's scoring (skips species with no positives).
- `mixup_batch` — blends pairs of embeddings + labels (Beta(0.4,0.4), p=0.5). Cheap free regularization since embeddings are already in RAM.
- `train_one_epoch_mixup` / `validate` — standard PyTorch loops.


In [ ]:
def macro_auc(y_true, y_pred):
    scores = []
    for c in range(y_true.shape[1]):
        col = y_true[:, c]
        if col.sum() == 0 or col.sum() == len(col):
            continue
        try:
            scores.append(roc_auc_score(col, y_pred[:, c]))
        except ValueError:
            continue
    return float(np.mean(scores)) if scores else float("nan")


def mixup_batch(emb, label, alpha=0.4, p=0.5):
    if np.random.random() < p:
        lam  = float(np.random.beta(alpha, alpha))
        perm = torch.randperm(emb.size(0), device=emb.device)
        emb   = lam * emb   + (1 - lam) * emb[perm]
        label = lam * label + (1 - lam) * label[perm]
    return emb, label


def train_one_epoch_mixup(model, loader, optimizer, loss_fn, alpha=0.4, p=0.5):
    model.train()
    total, n = 0.0, 0
    for emb, label in loader:
        emb, label = emb.to(DEVICE), label.to(DEVICE)
        emb, label = mixup_batch(emb, label, alpha=alpha, p=p)
        optimizer.zero_grad()
        logits = model(emb)
        loss = loss_fn(logits, label)
        loss.backward()
        optimizer.step()
        total += loss.item(); n += 1
    return total / max(1, n)


@torch.no_grad()
def validate(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    for emb, label in loader:
        logits = model(emb.to(DEVICE)).cpu().numpy()
        all_logits.append(logits); all_labels.append(label.numpy())
    y_pred = np.concatenate(all_logits)
    y_true = np.concatenate(all_labels)
    y_prob = 1.0 / (1.0 + np.exp(-y_pred))
    return macro_auc(y_true, y_prob)


## 6. Build base train/val split (used by Phase 4)

Phase 4 trains 5 seeds on a **fixed** 70/30 file split. Phase 6 (next section)
uses 5 different K-fold splits.


In [ ]:
BATCH_SIZE          = 256
NUM_WORKERS         = 0
SOUNDSCAPE_REPLICAS = 10
EPOCHS              = 30
LR                  = 5e-4
WD                  = 1e-4

# Filter clips to species the model knows
clip_keep      = clip_index["primary_label"].isin(label_to_idx).to_numpy()
clip_spatial_tr = clip_spatial[clip_keep]
clip_idx_train  = clip_index[clip_keep].reset_index(drop=True)
clip_ds_full   = SpatialEmbDataset(clip_spatial_tr, clip_idx_train, clip_label_vec)

# Fixed Phase 4 file split
unique_files = sorted(ss_index["filename"].unique())
shuffled_p4  = np.random.default_rng(SEED).permutation(unique_files)
n_val_files  = max(1, int(len(shuffled_p4) * 0.3))
p4_val_files = set(shuffled_p4[:n_val_files])

ss_train_mask_p4 = (~ss_index["filename"].isin(p4_val_files)).to_numpy()
ss_val_mask_p4   = ss_index["filename"].isin(p4_val_files).to_numpy()
ss_train_p4_spatial = ss_spatial[ss_train_mask_p4]
ss_train_p4_idx     = ss_index[ss_train_mask_p4].reset_index(drop=True)
ss_val_p4_spatial   = ss_spatial[ss_val_mask_p4]
ss_val_p4_idx       = ss_index[ss_val_mask_p4].reset_index(drop=True)

ss_ds_p4   = SpatialEmbDataset(ss_train_p4_spatial, ss_train_p4_idx, ss_label_vec)
ss_rep_p4  = ConcatDataset([ss_ds_p4] * SOUNDSCAPE_REPLICAS)
train_ds_p4 = ConcatDataset([clip_ds_full, ss_rep_p4])
val_ds_p4   = SpatialEmbDataset(ss_val_p4_spatial, ss_val_p4_idx, ss_label_vec)
val_loader_p4 = DataLoader(val_ds_p4, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Phase 4 split: {len(unique_files)-n_val_files} train files / {n_val_files} val files")
print(f"  Train: {len(clip_ds_full):,} clips + {len(ss_rep_p4):,} SS (×10) = {len(train_ds_p4):,}")
print(f"  Val:   {len(val_ds_p4):,} SS windows")


## 7. Phase 4 — train 5 seeds on the fixed split

Each seed varies head initialization, batch shuffle order, dropout mask, and
mixup random choices. Saves `model_v8_seed{42..46}.pt`.

~2 minutes per seed × 5 = ~10 min total.


In [ ]:
SEEDS = [42, 43, 44, 45, 46]
loss_fn = nn.BCEWithLogitsLoss()

phase4_history = {}
phase4_best    = {}

for seed in SEEDS:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

    head = PerchSEDHead(EMBED_DIM, hidden_dim=512, num_classes=NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WD)
    train_loader = DataLoader(train_ds_p4, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

    best_ss_auc = -1.0
    ckpt_path   = CKPT_DIR / f"model_v8_seed{seed}.pt"
    history     = []

    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch_mixup(head, train_loader, optimizer, loss_fn)
        ss_auc = validate(head, val_loader_p4)
        history.append((epoch, train_loss, ss_auc))
        if ss_auc > best_ss_auc:
            best_ss_auc = ss_auc
            torch.save({
                "state_dict":   head.state_dict(),
                "species":      species,
                "label_to_idx": label_to_idx,
                "num_classes":  NUM_CLASSES,
                "embed_dim":    EMBED_DIM,
                "head_config":  {"hidden_dim": 512, "dropout": 0.3},
                "seed":         seed,
                "phase":        "4_sed",
                "head_class":   "PerchSEDHead",
            }, ckpt_path)

    dt = time.time() - t0
    phase4_history[seed] = history
    phase4_best[seed]    = best_ss_auc
    print(f"Phase 4 seed {seed}: best val_auc={best_ss_auc:.4f}  ({dt:.1f}s)")

print(f"\nPhase 4 mean best: {np.mean(list(phase4_best.values())):.4f}  std: {np.std(list(phase4_best.values())):.4f}")


## 8. Phase 6 — K-fold ensemble (5 file splits)

Splits the 66 labeled SS files into 5 folds. For each fold, train on 4 folds' worth
of files (≈53), validate on 1 fold's worth (≈13). All 35k clips used in every fold.

This adds **file-level diversity** to complement Phase 4's seed-level diversity.
Saves `model_v9_fold{0..4}.pt`.

~2.5 minutes per fold × 5 = ~12 min total.


In [ ]:
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=False)

# Build 5 folds using the same shuffled file ordering
shuffled_p6 = np.random.default_rng(SEED).permutation(unique_files)
folds = []
file_arr = np.array(shuffled_p6)
for fold_idx, (train_idx, val_idx) in enumerate(kf.split(file_arr)):
    folds.append((set(file_arr[train_idx]), set(file_arr[val_idx])))
    print(f"Fold {fold_idx}: train={len(folds[-1][0])} files, val={len(folds[-1][1])} files")

phase6_best = {}
for fold_idx, (train_files, val_files) in enumerate(folds):
    ss_train_mask = ss_index["filename"].isin(train_files).to_numpy()
    ss_val_mask   = ss_index["filename"].isin(val_files).to_numpy()

    ss_train_spatial = ss_spatial[ss_train_mask]
    ss_train_idx     = ss_index[ss_train_mask].reset_index(drop=True)
    ss_val_spatial   = ss_spatial[ss_val_mask]
    ss_val_idx       = ss_index[ss_val_mask].reset_index(drop=True)

    ss_ds   = SpatialEmbDataset(ss_train_spatial, ss_train_idx, ss_label_vec)
    ss_rep  = ConcatDataset([ss_ds] * SOUNDSCAPE_REPLICAS)
    train_ds = ConcatDataset([clip_ds_full, ss_rep])
    val_ds   = SpatialEmbDataset(ss_val_spatial, ss_val_idx, ss_label_vec)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    fold_seed = SEED + fold_idx
    random.seed(fold_seed); np.random.seed(fold_seed)
    torch.manual_seed(fold_seed); torch.cuda.manual_seed_all(fold_seed)

    head = PerchSEDHead(EMBED_DIM, hidden_dim=512, num_classes=NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WD)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

    best_ss_auc = -1.0
    ckpt_path   = CKPT_DIR / f"model_v9_fold{fold_idx}.pt"

    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch_mixup(head, train_loader, optimizer, loss_fn)
        ss_auc = validate(head, val_loader)
        if ss_auc > best_ss_auc:
            best_ss_auc = ss_auc
            torch.save({
                "state_dict":   head.state_dict(),
                "species":      species,
                "label_to_idx": label_to_idx,
                "num_classes":  NUM_CLASSES,
                "embed_dim":    EMBED_DIM,
                "head_config":  {"hidden_dim": 512, "dropout": 0.3},
                "fold":         fold_idx,
                "phase":        "6_kfold",
                "head_class":   "PerchSEDHead",
            }, ckpt_path)
    dt = time.time() - t0
    phase6_best[fold_idx] = best_ss_auc
    print(f"Phase 6 fold {fold_idx}: best val={best_ss_auc:.4f}  ({dt:.1f}s)  (val set is different per fold — direct comparison meaningless)")

print(f"\nPhase 6 mean: {np.mean(list(phase6_best.values())):.4f}  (per-fold val sets differ, so this is rougher than Phase 4)")


## 9. Combined ensemble evaluation on Phase 4 val set

Load all 10 saved checkpoints (5 Phase 4 seeds + 5 Phase 6 folds), evaluate as
a single ensemble on Phase 4's fixed val set.

**Note**: this evaluation is slightly biased *upward* — 4 of the 5 Phase 6 fold models
saw most of the Phase 4 val files during their own training. The unbiased
comparison is the Kaggle leaderboard.


In [ ]:
def load_head(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    h = PerchSEDHead(EMBED_DIM, **ckpt["head_config"], num_classes=NUM_CLASSES).to(DEVICE)
    h.load_state_dict(ckpt["state_dict"])
    h.eval()
    return h


seed_heads = [load_head(CKPT_DIR / f"model_v8_seed{s}.pt") for s in SEEDS]
fold_heads = [load_head(CKPT_DIR / f"model_v9_fold{i}.pt") for i in range(N_FOLDS)]
combined_heads = seed_heads + fold_heads

print(f"Loaded {len(combined_heads)} models  ({len(seed_heads)} seeds + {len(fold_heads)} folds)")

# Evaluate ensemble on Phase 4 val
@torch.no_grad()
def evaluate_ensemble(heads, loader):
    all_logits = None
    all_labels = []
    for emb, label in loader:
        emb = emb.to(DEVICE)
        batch_logits = sum(h(emb) for h in heads) / len(heads)
        chunk = batch_logits.cpu().numpy()
        all_logits = chunk if all_logits is None else np.concatenate([all_logits, chunk])
        all_labels.append(label.numpy())
    y_true = np.concatenate(all_labels)
    y_prob = 1.0 / (1.0 + np.exp(-all_logits))
    return macro_auc(y_true, y_prob)


seed_only_auc     = evaluate_ensemble(seed_heads, val_loader_p4)
fold_only_auc     = evaluate_ensemble(fold_heads, val_loader_p4)
combined_auc      = evaluate_ensemble(combined_heads, val_loader_p4)

print(f"\n5-seed ensemble (Phase 4 only):    {seed_only_auc:.4f}")
print(f"5-fold ensemble (Phase 6 only, BIASED on Phase 4 val): {fold_only_auc:.4f}")
print(f"Combined 10-model ensemble:        {combined_auc:.4f}")
print(f"\nLB reference:")
print(f"  Phase 4 alone:    LB 0.839")
print(f"  Phase 6 alone:    LB 0.839")
print(f"  Combined:         LB 0.842 (CURRENT BEST)")


## 10. Plot training curves

Per-seed val curves for Phase 4 (Phase 6 omitted — different val sets, can't overlay).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for seed, hist in phase4_history.items():
    ep, loss, auc = zip(*hist)
    axes[0].plot(ep, loss, alpha=0.6, label=f"seed {seed}")
    axes[1].plot(ep, auc,  alpha=0.6, label=f"seed {seed}")
axes[0].set_title("Phase 4 train loss (5 seeds)"); axes[0].set_xlabel("epoch")
axes[0].legend(fontsize=8)
axes[1].set_title("Phase 4 val_auc (5 seeds)"); axes[1].set_xlabel("epoch")
axes[1].axhline(combined_auc, color="purple", linestyle="-", linewidth=2, label=f"Combined ensemble ({combined_auc:.4f})")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()


## What's next

When training completes:
- All 10 checkpoints live in `checkpoints/model_v8_seed*.pt` + `checkpoints/model_v9_fold*.pt`.
- Open **`02_infer.ipynb`** to generate `submission.csv`.
- Or upload checkpoints as a Kaggle dataset and submit via the Kaggle kernel (see `kaggle-uploads/baseline-submit/`).

For further gains, see `ROADMAP.md` — the next levers are Phase 7 (calibration) and Phase 8 (SSL / novel ideas).
